In [11]:
# --- MÓDULO DE INGESTÃO: DINÂMICA DEMOGRÁFICA (IBGE) ---

import pandas as pd

# Defino o ponto de entrada para os dados de população residente.
# Esta variável é o denominador crítico para a construção de todos os 
# indicadores "per capita" do modelo econométrico.
caminho_arquivo = r'C:\Users\fabio\TCC\3_Populacao.csv'

try:
    # Realizo a leitura do dataset demográfico.
    # skiprows=1: Ignora o título institucional do relatório do IBGE.
    # encoding='utf-8-sig': Trata o Byte Order Mark (BOM) comum em arquivos CSV brasileiros.
    df_populacao = pd.read_csv(
        caminho_arquivo,
        skiprows=1,
        sep=';',
        encoding='utf-8-sig'
    )
    
    # DIAGNÓSTICO ESTRUTURAL:
    # Verificação das primeiras observações para validar o alinhamento das colunas.
    print("--- Visualização do Dataset Demográfico (IBGE) ---")
    print(df_populacao.head())
    
    print("\n--- Inventário de Atributos Disponíveis ---")
    print(df_populacao.columns.tolist())

except Exception as e:
    print(f"ERRO DE INGESTÃO: Falha ao processar o arquivo de população: {e}")

--- TABELA CARREGADA ---
  Sigla   Código     Município     1992     1993     1994     1995     1997  \
0    AC  1200013    Acrelândia   5688.0   5858.0   6036.0   6211.0   6536.0   
1    AC  1200054  Assis Brasil   3017.0   3182.0   3302.0   3419.0   2918.0   
2    AC  1200104     Brasiléia  14353.0  14737.0  15076.0  15406.0  13946.0   
3    AC  1200138        Bujari   3294.0   3401.0   3501.0   3599.0   4392.0   
4    AC  1200179      Capixaba   2367.0   2418.0   2465.0   2512.0   3109.0   

      1998     1999  ...     2014     2015     2016     2017     2018  \
0   6730.0   6922.0  ...  13613.0  13869.0  14120.0  14366.0  15020.0   
1   2918.0   2919.0  ...   6610.0   6738.0   6863.0   6986.0   7300.0   
2  13938.0  13930.0  ...  23378.0  23849.0  24311.0  24765.0  25848.0   
3   4641.0   4888.0  ...   9173.0   9339.0   9503.0   9664.0  10111.0   
4   3286.0   3460.0  ...  10170.0  10498.0  10820.0  11136.0  11456.0   

      2019     2020     2021     2024  Unnamed: 31  
0  15256

In [12]:
# --- ETAPA 2: HIGIENIZAÇÃO E REESTRUTURAÇÃO DEMOGRÁFICA (TIDY DATA) ---

# Nesta etapa, realizo a limpeza de artefatos de exportação e a transposição do 
# dataset para o formato 'Long'. Esta estrutura é o padrão "Tidy Data", 
# fundamental para realizar junções (merges) com as séries financeiras e de saúde.

print("Iniciando processo de higienização de colunas...")

# 1. REMOÇÃO DE ATRIBUTOS RESIDUAIS
# Trato a coluna 'Unnamed: 31', um artefato comum em exportações do SIDRA/IBGE 
# causado por delimitadores sobressalentes ao final das linhas.
try:
    df_populacao_limpa = df_populacao.drop(columns=['Unnamed: 31'])
    print("Sucesso: Atributo residual 'Unnamed: 31' removido.")
except KeyError:
    print("Aviso: Atributo residual não detectado; prosseguindo com a estrutura atual.")
    df_populacao_limpa = df_populacao.copy()

# 2. REESTRUTURAÇÃO PARA FORMATO DE PAINEL (MELT)
# Transponho a matriz de 'Wide' para 'Long', consolidando os anos em uma única 
# variável temporal. Isso viabiliza o controle de séries históricas no Stata.

# 2a. Definição das Dimensões Fixas (Chaves Primárias)
eixos_identificacao = ['Sigla', 'Código', 'Município']

# 2b. Execução do Reshaping
# Converto as colunas de anos em observações verticais, criando as variáveis 'Ano' e 'Populacao'.
df_populacao_final = df_populacao_limpa.melt(
    id_vars=eixos_identificacao,
    var_name='Ano',
    value_name='Populacao'
)

# 3. AUDITORIA DE ESTRUTURA FINAL
# Verifico os extremos do novo dataset para garantir a integridade da transposição.
print("\n--- Diagnóstico de Estrutura: Painel Demográfico (Long Format) ---")
print(df_populacao_final.head())
print(df_populacao_final.tail())

Coluna 'Unnamed: 31' removida com sucesso.

--- Tabela 'Derretida' (Formato Longo) ---
  Sigla   Código     Município   Ano  Populacao
0    AC  1200013    Acrelândia  1992     5688.0
1    AC  1200054  Assis Brasil  1992     3017.0
2    AC  1200104     Brasiléia  1992    14353.0
3    AC  1200138        Bujari  1992     3294.0
4    AC  1200179      Capixaba  1992     2367.0
       Sigla   Código       Município   Ano  Populacao
156711    TO  1721208  Tocantinópolis  2024    23203.0
156712    TO  1721257        Tupirama  2024     2003.0
156713    TO  1721307      Tupiratins  2024     1897.0
156714    TO  1722081    Wanderlândia  2024    10751.0
156715    TO  1722107         Xambioá  2024    10683.0


In [13]:
# --- ETAPA 3: SERIALIZAÇÃO E PERSISTÊNCIA DO DATASET DEMOGRÁFICO (OUTPUT) ---

# Concluo o processamento da variável populacional exportando o dataset 
# no formato 'Long'. Este arquivo é o componente base para a normalização 
# de todas as variáveis dependentes e independentes do modelo (Análise Per Capita).

# 1. DEFINIÇÃO DO REPOSITÓRIO DE SAÍDA
# Armazeno o arquivo final com nomenclatura clara para garantir a 
# rastreabilidade durante a fase de integração mestre (Master Merge).
caminho_saida_csv = r'C:\Users\fabio\TCC\3_populacao_FINAL.csv'

# 2. EXPORTAÇÃO E INTEGRIDADE ESTRUTURAL
# Utilizo o parâmetro index=False para assegurar que o CSV contenha 
# apenas as chaves primárias e a métrica populacional, otimizando o consumo de memória.
df_populacao_final.to_csv(
    caminho_saida_csv,
    index=False
)

print(f"--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---")
print(f"Dataset Demográfico Consolidado: {caminho_saida_csv}")

--- SUCESSO! ---
Seu arquivo de população consolidado foi salvo em:
C:\Users\fabio\TCC\3_populacao_FINAL.csv
